In [42]:
import pandas as pd

df = pd.read_json("dados/noticias_brutas.json")
df.head()

,id,titulo,texto,data,fonte
0,1,"Copom mantém Selic em 13,75% ao ano pela quart...",<p>Publicado em: 02/08/2023 às 20h15</p>\n\n\n...,2023-08-02,Banco Central do Brasil
1,2,"IPCA de julho registra 0,12%, menor resultado ...",Publicado em: 09/08/2023 &agrave;s 09h00\n\n\n...,2023-08-09,IBGE
2,3,"PIB do Brasil cresce 1,9% no segundo trimestre...",<p><em>Publicado em: 01/09/2023 &agrave;s 10h3...,2023-09-01,IBGE
3,4,"Taxa de desemprego cai para 7,9% no segundo tr...",Publicado em: 29/08/2023\n\n\n\n<p>A taxa de d...,2023-08-29,IBGE
4,5,"Dólar fecha abaixo de R$ 4,80 com expectativa ...",<p>23/08/2023 - 18h45 | Mercados</p>\n\n\n<p>O...,2023-08-23,Valor Econômico


# BIBLIOTECAS


In [43]:
!pip install beautifulsoup4 sentence-transformers scikit-learn

In [44]:
# Bibliotecas
import re
import html
from bs4 import BeautifulSoup

# Etapa 1 — Limpeza e Tratamento de Texto

In [45]:
# Função que limpa os textos

def limpar_texto(texto):
    if not isinstance(texto, str):
        return ""

    texto = html.unescape(texto)

    soup = BeautifulSoup(texto, "html.parser")
    texto = soup.get_text(separator=" ")

    texto = re.sub(
        r"Publicado em:\s*\d{2}/\d{2}/\d{4}(?:\s*às\s*\d{2}h\d{2})?",
        " ",
        texto
    )

    texto = re.sub(
        r"\d{2}/\d{2}/\d{4}\s*[-—]?\s*\d{0,2}h?\d{0,2}?\s*\|?\s*\w*",
        " ",
        texto
    )

    texto = re.sub(r"\b\d{2}/\d{2}/\d{4}\b", " ", texto)
    texto = re.sub(r"https?://\S+|www\.\S+", " ", texto)
    texto = re.sub(r"\s+", " ", texto)

    return texto.strip()


In [46]:
# Limpeza no DataFrame

df["texto_limpo"] = df["texto"].apply(limpar_texto)
df[["titulo", "texto", "texto_limpo"]].head(3)

,titulo,texto,texto_limpo
0,"Copom mantém Selic em 13,75% ao ano pela quart...",<p>Publicado em: 02/08/2023 às 20h15</p>\n\n\n...,O Comitê de Política Monetária (Copom) do Banc...
1,"IPCA de julho registra 0,12%, menor resultado ...",Publicado em: 09/08/2023 &agrave;s 09h00\n\n\n...,O Índice Nacional de Preços ao Consumidor Ampl...
2,"PIB do Brasil cresce 1,9% no segundo trimestre...",<p><em>Publicado em: 01/09/2023 &agrave;s 10h3...,O Produto Interno Bruto (PIB) do Brasil cresce...


In [47]:
# Tratar o texto "Selic."

df["tamanho_texto"] = df["texto_limpo"].str.len()
df[df["tamanho_texto"] < 30]

,id,titulo,texto,data,fonte,texto_limpo,tamanho_texto
17,18,Nota curta,Selic.,2023-08-01,Desconhecida,Selic.,6


In [48]:
# Salvar dados limpos

df.to_csv("dados/noticias_limpas.csv", index=False, encoding="utf-8-sig")
df[["titulo", "texto_limpo"]].head()

,titulo,texto_limpo
0,"Copom mantém Selic em 13,75% ao ano pela quart...",O Comitê de Política Monetária (Copom) do Banc...
1,"IPCA de julho registra 0,12%, menor resultado ...",O Índice Nacional de Preços ao Consumidor Ampl...
2,"PIB do Brasil cresce 1,9% no segundo trimestre...",O Produto Interno Bruto (PIB) do Brasil cresce...
3,"Taxa de desemprego cai para 7,9% no segundo tr...","A taxa de desemprego no Brasil recuou para 7,9..."
4,"Dólar fecha abaixo de R$ 4,80 com expectativa ...",| Mercados O dólar comercial encerrou a sessão...


# Etapa 2 — Geração de Embeddings

In [49]:
# Importar modelo

from sentence_transformers import SentenceTransformer

In [50]:
# Carreguei esse modelo por ter um bom desempenho para similaridade semântica em português,
# além de ser relativamente leve e fácil de reproduzir localmente.

modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [51]:
# Gerar embeddings dos textos

embeddings = modelo.encode(
    df["texto_limpo"].tolist(),
    convert_to_numpy=True
)
embeddings.shape

(20, 384)

# Etapa 3 — Motor de Busca Semântico

In [52]:
# Importar similaridade

from sklearn.metrics.pairwise import cosine_similarity

# Função de busca

def buscar_noticias(consulta, top_k=3):
    consulta_embedding = modelo.encode(
        [consulta],
        convert_to_numpy=True
    )

    similaridades = cosine_similarity(
        consulta_embedding,
        embeddings
    )[0]

    df_resultado = df.copy()
    df_resultado["similaridade"] = similaridades

    return df_resultado.sort_values(
        by="similaridade",
        ascending=False
    ).head(top_k)

In [53]:
# Texto ponderado
df["texto_busca"] = (
    df["titulo"] + " " +
    df["titulo"] + " " +
    df["texto_limpo"]
)


In [54]:
# embeddings

embeddings = modelo.encode(
    df["texto_busca"].tolist(),
    convert_to_numpy=True
)

In [55]:
# mudanças na taxa de juros

resultado_juros = buscar_noticias("mudanças na taxa de juros")
resultado_juros[["titulo", "similaridade"]]


,titulo,similaridade
10,"Selic deve recuar a 9% até o fim de 2024, proj...",0.593734
14,Câmbio: real se fortalece com melhora do ambie...,0.584740
5,Copom inicia ciclo de corte e reduz Selic para...,0.573734


In [56]:
# mercado de trabalho e desemprego

resultado_emprego = buscar_noticias("mercado de trabalho e desemprego")
resultado_emprego[["titulo", "similaridade"]]

,titulo,similaridade
3,"Taxa de desemprego cai para 7,9% no segundo tr...",0.638973
13,Desemprego juvenil no Brasil ainda preocupa ap...,0.608060
6,"Inadimplência das famílias sobe para 6,3% em j...",0.447606


In [57]:
# inflação e preços ao consumidor

resultado_inflacao = buscar_noticias("inflação e preços ao consumidor")
resultado_inflacao[["titulo", "similaridade"]]


,titulo,similaridade
8,Inflação ao produtor (IPA) desacelera e pressã...,0.660301
14,Câmbio: real se fortalece com melhora do ambie...,0.594034
10,"Selic deve recuar a 9% até o fim de 2024, proj...",0.578075
